# Fine tuning a masked language model

In [1]:
from datasets import load_dataset 
import pandas as pd 
imdb_dataset = load_dataset('imdb')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

**Dataset overview**

In [3]:
imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

checking the labels for unsupervised split it contains **-1** as label

In [2]:
sample = imdb_dataset['unsupervised'].shuffle().select(range(1))
for row in sample:
    print(row['text'])
    print(row['label'])

Oh yes European culture and the Eurovision Song Contest which is like Adolph Hitler and racial harmony appearing in the same sentence . Actually this years contest was far worse than usual because the tunes seemed confused . Croatia's song started off with some bagpipe music while the entries from Greece and the United Kingdom sounded more Turkish than the entry by the Turks . Oh and the Greeks did a bit of riverdancing which probably was a cynical attempt to fool the voters into thinking they were watching Ireland who often win but for some reason didn't qualify for the final this year <br /><br />Okay lets be truthful and say I was hoping the best looking female contestant was going to win . I guess I shouldn't be too disappointed since the Greek contestant had the best legs and she won while Romania had the best cleavage and she came third with the Israeli motek also doing very well . Unfortunately there was a miscarriage of justice with some ugly fat bird from Malta getting the run

In [3]:
sample = imdb_dataset['train'].shuffle().select(range(5))
for row in sample:
    print(f"Sample : {row['text']}")
    print(f"Label : {row['label']}")

pd.Series(imdb_dataset['train']['label'][:]).unique()

Sample : Kusturika made it again. Another masterpiece. A coral comedy full of his own landmarks, with a frenetic rhythm and many glorious moments, we laughed and laughed, what a party! The music is everywhere, and also the shooting, the animals, the crazy bastards, sex and amazing gadgets and inventions, everything colorfully visual to entertain only. Pure cinema in essence. A wonderful experience to watch. And one is specially grateful since good comedies are so rare, and so wonderful. Well, this is one, and if you enjoyed Kusturica's previous films, you'll love this, although, as in all comedies, it is about a chemical reaction, and you have to be in the mood for it.
Label : 1
Sample : Hello Dave Burning Paradise is a film for anyone who likes Jackie Chan and Indiana Jones. The films main protagonist is most definitely the bastard son of these two strange fathers. As for the other characters well they are familiar transformations of similar action film stereotypes. Where this film is

array([0, 1])

**Initializing Tokenizer and model**

In [4]:
from transformers import AutoModelForMaskedLM,AutoTokenizer
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForMaskedLM.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

**Preprocessing dataset**

In [5]:
def tokenize_function(examples):
    result = tokenizer(examples['text'])
    if tokenizer.is_fast:
        result['word_ids'] = [result.word_ids(i) for i in range(len(result['input_ids']))]
    return result 

tokenized_datasets = imdb_dataset.map(tokenize_function,batched=True,remove_columns=['text','label'])
tokenized_datasets

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (720 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 50000
    })
})

**Grouping is done in order to use gpu efficiently**

In [8]:
chunk_size = 128
def group_texts(example):
    concatenated_examples = {
        k : sum(example[k],[]) for k in example.keys()
    }
    total_length = len(concatenated_examples[list(example.keys())[0]])
    total_length = (total_length // chunk_size) * chunk_size 

    results = {
        k : [t[i:i+chunk_size] for i in range(0,total_length,chunk_size)]
        for k,t in concatenated_examples.items()
    }

    results['label'] = results['input_ids'].copy()
    return results

In [9]:
preprocessed_dataset = tokenized_datasets.map(group_texts,batched=True)
preprocessed_dataset

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'label'],
        num_rows: 61291
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'label'],
        num_rows: 59904
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'label'],
        num_rows: 122957
    })
})

In [10]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=0.15)

In [11]:
samples = [preprocessed_dataset['train'][i] for i in range(2)]
for sample in samples:
    _ = sample.pop('word_ids')

for chunk in data_collator(samples)["input_ids"]:
    print(f"\n {tokenizer.decode(chunk)}")


 [CLS] [MASK] [MASK] [MASK] am curious - yellow from my video store because of all the controversy that surrounded it when [MASK] was first released in 1967. i also [MASK] that at first [MASK] was seized by u. メ. customs if it ever tried to enter this country, therefore being a [MASK] of films considered " controversial " i really had to [MASK] this for myself. < br / > < douglas / > the [MASK] is centered [MASK] a [MASK] swedish [MASK] student named [MASK] who [MASK] to learn everything she [MASK] about life [MASK] in particular she wants to focus her attentions to making some sort shield [MASK] on what the average swede thought about [MASK] political issues such

 [MASK] the vietnam war and race [MASK] in the [MASK] states. in between asking [MASK] and ordinary [MASK]izens of stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men [MASK] [MASK] br / > [MASK] br / > what kills me aboutttered [MASK] [MASK] - yellow [MASK] that 40 yea

In [12]:
train_size = 10000
test_size = int(0.1*train_size)

downsampled_dataset = preprocessed_dataset['train'].train_test_split(
    train_size = train_size,
    test_size = test_size,
    seed=45
)

downsampled_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'label'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'label'],
        num_rows: 1000
    })
})

**Training pipeline using Trainer and Training Arguments**

In [13]:
from transformers import TrainingArguments,Trainer
batch_size = 32

logging_steps = len(downsampled_dataset['train'])//batch_size 


training_args = TrainingArguments(
    eval_strategy = "epoch", # evaluation_strategy deprecated
    learning_rate = 0.00002,
    weight_decay = 0.01,
    per_device_train_batch_size = batch_size,
    per_device_eval_batch_size = batch_size,
    push_to_hub=False,
    fp16 = True,
    logging_steps=logging_steps
)

In [14]:
trainer = Trainer(
    model = model,
    train_dataset= downsampled_dataset['train'],
    eval_dataset=downsampled_dataset['test'],
    args = training_args,
    data_collator = data_collator,
    # tokenizer = tokenizer 
)

In [15]:
import math

eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

>>> Perplexity: 21.96


In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,2.673666,2.463044,0.001600
2,2.542297,2.437812,0.001600
3,2.510511,2.351555,0.001600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=939, training_loss=2.5754358689751995, metrics={'train_runtime': 169.4711, 'train_samples_per_second': 177.021, 'train_steps_per_second': 5.541, 'total_flos': 994208670720000.0, 'train_loss': 2.5754358689751995, 'epoch': 3.0})

In [17]:
eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

>>> Perplexity: 11.20


**Training pipeline using Pytorch**

In [25]:
import torch
from torch.utils.data import DataLoader 

train_loader = DataLoader(
    downsampled_dataset['train'],
    shuffle=True,
    batch_size=batch_size,
    collate_fn=data_collator
)

eval_loader = DataLoader(
    downsampled_dataset['test'],
    shuffle = False,
    batch_size = batch_size,
    collate_fn=data_collator
)

In [23]:
model = AutoModelForMaskedLM.from_pretrained(checkpoint)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [27]:
from transformers import get_scheduler 
optimizer = torch.optim.AdamW(model.parameters(),lr=2e-5)
num_train_epochs = 3 
num_update_steps_per_epoch = len(train_loader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch 
lr_scheduling = get_scheduler(
    "linear",
    optimizer = optimizer,
    num_warmup_steps = 0,
    num_training_steps=num_training_steps
)
